# Question 9: Genetic Algorithm for the Traveling Salesman Problem (TSP)

**Dataset:** Oliver30 (`oliver30.tsp`)

This notebook follows the workflow of the Q1 Genetic Algorithm notebook: initialize a population, calculate tour fitness and selection probabilities, compute expected/actual counts, create a mating pool, apply crossover and mutation, then evaluate convergence.

**Before running:** Place `oliver30.tsp` in `notebooks/data/` (or `data/` if the notebook is run from inside the `notebooks/` directory). The loader checks both locations.

In [ ]:
import random
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility
random.seed(42)
np.random.seed(42)

# ============================================================
# 1. LOAD OLIVER30 DATASET
# ============================================================

def load_tsp(filename):
    cities = []
    reading = False
    edge_weight_type = None

    with open(filename, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            if line.startswith("EDGE_WEIGHT_TYPE"):
                edge_weight_type = line.split(":", 1)[-1].strip()

            if line == "NODE_COORD_SECTION":
                reading = True
                continue

            if line == "EOF":
                break

            if reading and line:
                parts = line.split()
                cities.append([float(parts[1]), float(parts[2])])

    if not cities:
        raise ValueError(
            "No coordinates found. Confirm the file contains NODE_COORD_SECTION."
        )

    return np.array(cities), edge_weight_type


candidates = [
    Path("data/oliver30.tsp"),
    Path("notebooks/data/oliver30.tsp"),
]

tsp_path = next((p for p in candidates if p.exists()), None)

if tsp_path is None:
    raise FileNotFoundError(
        "oliver30.tsp not found. Put it in data/ or notebooks/data/ "
        "relative to the current working directory."
    )

cities, edge_weight_type = load_tsp(tsp_path)
N = len(cities)

print("Dataset:", tsp_path.resolve())
print("Number of cities:", N)
print("EDGE_WEIGHT_TYPE:", edge_weight_type)

# ============================================================
# 2. CREATE DISTANCE MATRIX
# ============================================================

distance = np.zeros((N, N), dtype=float)

for i in range(N):
    for j in range(N):
        d = np.linalg.norm(cities[i] - cities[j])

        # TSPLIB EUC_2D uses nearest-integer Euclidean distances.
        if edge_weight_type == "EUC_2D":
            distance[i, j] = math.floor(d + 0.5)
        else:
            # Fallback: ordinary Euclidean distance.
            distance[i, j] = d

# ============================================================
# 3. GA PARAMETERS
# ============================================================

POP_SIZE = 100
GENERATIONS = 500
MUTATION_RATE = 0.10
ELITE_SIZE = 2
TOURNAMENT_SIZE = 5

# ============================================================
# 4. CHROMOSOME AND TOUR FITNESS
# ============================================================

# Each chromosome is a permutation of city indices 0, ..., N-1.
def create_route():
    return random.sample(range(N), N)


def tour_distance(route):
    return sum(
        distance[route[i], route[(i + 1) % N]]
        for i in range(N)
    )


# ============================================================
# 5. FITNESS, PROBABILITY, EXPECTED COUNT, ACTUAL COUNT
# ============================================================

def build_generation_table(population):
    df = pd.DataFrame({
        "Chromosome": [route[:] for route in population]
    })

    df["Route (city IDs)"] = df["Chromosome"].apply(
        lambda route: [city + 1 for city in route]
    )
    df["Tour Distance"] = df["Chromosome"].apply(tour_distance)

    # TSP is a minimization problem, so smaller distance means
    # larger selection fitness.
    df["Selection Fitness"] = 1.0 / (df["Tour Distance"] + 1e-12)
    df["Probability"] = (
        df["Selection Fitness"] / df["Selection Fitness"].sum()
    )
    df["Expected Count"] = df["Probability"] * len(population)

    # Largest-remainder allocation ensures counts sum to POP_SIZE.
    df["Actual Count"] = np.floor(df["Expected Count"]).astype(int)
    remaining = len(population) - int(df["Actual Count"].sum())

    if remaining > 0:
        fractions = df["Expected Count"] - df["Actual Count"]
        for idx in fractions.nlargest(remaining).index:
            df.loc[idx, "Actual Count"] += 1

    return df


def mating_pool_from(table):
    pool = []
    for _, row in table.iterrows():
        pool.extend(
            [row["Chromosome"][:]] * int(row["Actual Count"])
        )
    random.shuffle(pool)
    return pool


# ============================================================
# 6. ORDERED CROSSOVER (OX)
# ============================================================

def crossover(parent1, parent2):
    left, right = sorted(random.sample(range(N), 2))
    child = [None] * N
    child[left:right + 1] = parent1[left:right + 1]

    remaining = [city for city in parent2 if city not in child]
    j = 0
    for i in range(N):
        if child[i] is None:
            child[i] = remaining[j]
            j += 1

    return child


# ============================================================
# 7. SWAP MUTATION
# ============================================================

def mutate(route):
    route = route[:]
    if random.random() < MUTATION_RATE:
        i, j = random.sample(range(N), 2)
        route[i], route[j] = route[j], route[i]
    return route


# ============================================================
# 8. INITIAL POPULATION
# ============================================================

population = [create_route() for _ in range(POP_SIZE)]
initial_table = build_generation_table(population)

print("Initial population (first 10 rows):")
display(initial_table.drop(columns=["Chromosome"]).head(10))
print("Initial best tour distance:", initial_table["Tour Distance"].min())


# ============================================================
# 9. EVOLUTION: SELECTION -> CROSSOVER -> MUTATION
# ============================================================

best_history = []
best_route = min(population, key=tour_distance)[:]
best_distance = tour_distance(best_route)

for generation in range(GENERATIONS):
    population.sort(key=tour_distance)

    # Track global best (best-so-far)
    if tour_distance(population[0]) < best_distance:
        best_route = population[0][:]
        best_distance = tour_distance(best_route)

    best_history.append(best_distance)

    # Calculate selection probabilities and actual counts.
    generation_table = build_generation_table(population)
    mating_pool = mating_pool_from(generation_table)

    # Elitism: retain top routes unchanged.
    elites = [route[:] for route in population[:ELITE_SIZE]]
    next_population = elites

    while len(next_population) < POP_SIZE:
        # Tournament selection from the mating pool.
        contestants1 = random.sample(
            mating_pool, min(TOURNAMENT_SIZE, len(mating_pool))
        )
        contestants2 = random.sample(
            mating_pool, min(TOURNAMENT_SIZE, len(mating_pool))
        )
        parent1 = min(contestants1, key=tour_distance)
        parent2 = min(contestants2, key=tour_distance)

        child = crossover(parent1, parent2)
        child = mutate(child)
        next_population.append(child)

    population = next_population

# Final best-so-far check
final_table = build_generation_table(population)
final_population_best = min(population, key=tour_distance)

if tour_distance(final_population_best) < best_distance:
    best_route = final_population_best[:]
    best_distance = tour_distance(best_route)

print("\nFinal population (first 10 rows):")
display(final_table.drop(columns=["Chromosome"]).head(10))

initial_best = float(initial_table["Tour Distance"].min())
print("\nCONVERGENCE REPORT")
print("-" * 45)
print("Initial population best distance:", initial_best)
print("Final population best distance:", tour_distance(final_population_best))
print("Best-so-far distance:", best_distance)
print("Improvement:", initial_best - best_distance)
print("Best tour (city IDs, 1-based):", [city + 1 for city in best_route])

assert sorted(best_route) == list(range(N))
print("Valid tour: every city is visited exactly once.")

# ============================================================
# 10. CONVERGENCE GRAPH
# ============================================================

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(best_history) + 1), best_history)
plt.xlabel("Generation")
plt.ylabel("Best-so-far tour distance")
plt.title("GA Convergence on Oliver30 TSP")
plt.grid(True)
plt.tight_layout()
plt.show()

# ============================================================
# 11. BEST TOUR VISUALIZATION
# ============================================================

closed_route = best_route + [best_route[0]]
route_coords = cities[closed_route]

plt.figure(figsize=(9, 7))
plt.plot(route_coords[:, 0], route_coords[:, 1], "o-")

for i, (x, y) in enumerate(cities):
    plt.annotate(str(i + 1), (x, y))

plt.title(f"Best TSP Tour Found (Distance = {best_distance:.2f})")
plt.xlabel("X Coordinate")
plt.ylabel("Y Coordinate")
plt.grid(True)
plt.tight_layout()
plt.show()
